# Phase 5 — Test Set: Córdoba Fires (Oct 2020)

**Environment**: Local  
**Estimated time**: 2–3 h (download) + 30 min (processing)  
**Output**: `data/cordoba/patches/images/` + `data/cordoba/patches/masks_dnbr/`

## Why Córdoba

The Córdoba (Argentina) fires of October 2020 burned ~350,000 ha across the Sierra Chica
and Comechingones ranges. Using Córdoba as the test set demonstrates **geographic generalization**:

- Training: Corrientes (northeast, wetlands + grasslands), Dec 2021–Feb 2022
- Test: Córdoba (central-west, sierras + xerophytic scrubland), Oct 2020
- Same model, different region → validates that it learned real spectral signatures

## dNBR Strategy for Córdoba

With two dates from the SAME tile (pre and post fire) we compute true dNBR:
```
dNBR = NBR(ago/sep 2020) − NBR(oct/nov 2020)
```
Threshold dNBR > 0.10 → burn scar


In [ ]:
import sys, os, time, re, warnings
from pathlib import Path

_conda_prefix = Path(sys.executable).parent.parent
_proj_data = _conda_prefix / 'Library' / 'share' / 'proj'
if _proj_data.exists():
    os.environ['PROJ_DATA'] = str(_proj_data)
    os.environ['PROJ_LIB']  = str(_proj_data)

import requests
import pystac_client
import numpy as np
import rasterio
import rasterio.warp
import rasterio.features
import rasterio.windows
from rasterio.crs import CRS
from dotenv import load_dotenv
from tqdm import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

load_dotenv(dotenv_path=Path('..') / '.env')
CDSE_USER     = os.getenv('CDSE_USER')
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD')

# Córdoba has its own folder — separate from Corrientes
CORDOBA_DIR  = Path('..') / 'data' / 'cordoba'
RAW_DIR      = CORDOBA_DIR / 'sentinel2' / 'raw'
PROC_DIR     = CORDOBA_DIR / 'sentinel2' / 'processed'
PATCH_DIR    = CORDOBA_DIR / 'patches'
for d in [RAW_DIR, PROC_DIR, PATCH_DIR / 'images', PATCH_DIR / 'masks_dnbr']:
    d.mkdir(parents=True, exist_ok=True)

print(f'CDSE user: {CDSE_USER}')
print(f'Output: {CORDOBA_DIR.resolve()}')

In [ ]:
# ── AOI y ventanas temporales ─────────────────────────────────────────────────
# Sierra Chica + Comechingones, Córdoba, Argentina
# Los incendios de 2020 se concentraron en el eje N-S de las sierras
BBOX = [-65.5, -33.0, -62.5, -30.5]  # [W, S, E, N]

TIME_WINDOWS = [
    ('2020-08-01T00:00:00Z', '2020-09-30T23:59:59Z', 'pre_fire'),    # antes del fuego
    ('2020-11-01T00:00:00Z', '2020-12-15T23:59:59Z', 'post_fire'),   # después del fuego
]

BANDS     = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12', 'SCL']
MAX_CLOUD = 15   # más estricto para Córdoba — zona montañosa con más nubes

print(f'BBOX Córdoba: {BBOX}')
print(f'Ventanas: pre_fire (Aug-Sep 2020) | post_fire (Nov-Dec 2020)')
print(f'Max cloud cover: {MAX_CLOUD}%')

In [ ]:
# ── Búsqueda STAC ─────────────────────────────────────────────────────────────
catalog = pystac_client.Client.open('https://catalogue.dataspace.copernicus.eu/stac')

all_items = []
for start, end, label in TIME_WINDOWS:
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=BBOX,
        datetime=f'{start}/{end}',
        max_items=100
    )
    items_raw = list(search.items())
    items = [it for it in items_raw
             if it.properties.get('eo:cloud_cover', 100) < MAX_CLOUD]
    for it in items:
        it.extra_fields['window_label'] = label
    all_items.extend(items)
    print(f'  {label:12} → {len(items_raw):3} total, {len(items):3} with cloud<{MAX_CLOUD}%')

print(f'\nTotal: {len(all_items)} items found')

In [ ]:
# ── Selección: 1 escena por tile por ventana temporal ─────────────────────────
# Para dNBR verdadero necesitamos el MISMO tile en pre y post.
# Seleccionamos tiles que tengan imágenes en AMBAS ventanas.

def tile_id_from_item(item):
    """Extracts the MGRS tile ID (ej. '20HML') from the STAC item."""
    return item.properties.get('s2:mgrs_tile', item.id.split('_T')[-1][:5])

by_window = defaultdict(list)
for item in all_items:
    by_window[item.extra_fields['window_label']].append(item)

# Tiles disponibles por ventana
tiles_pre  = {tile_id_from_item(it) for it in by_window.get('pre_fire', [])}
tiles_post = {tile_id_from_item(it) for it in by_window.get('post_fire', [])}
common_tiles = tiles_pre & tiles_post

print(f'Tiles with pre+post: {sorted(common_tiles)}')
print(f'Common tiles: {len(common_tiles)}')

# Para cada tile común, tomar la imagen de menor cloud cover en cada ventana
selected = []
for tile in sorted(common_tiles):
    for label, window_items in by_window.items():
        tile_items = [it for it in window_items if tile_id_from_item(it) == tile]
        if tile_items:
            best = min(tile_items, key=lambda x: x.properties.get('eo:cloud_cover', 100))
            selected.append(best)
            date  = best.datetime.strftime('%Y-%m-%d')
            cloud = best.properties.get('eo:cloud_cover', '?')
            print(f'  {label:12} | tile {tile} | {date} | cloud={cloud:.1f}% | {best.id[:45]}')

print(f'\nScenes to download: {len(selected)}')

In [ ]:
# ── Descarga de bandas (mismo patrón que 01_download_sentinel2.ipynb) ─────────
def get_cdse_token(user, password):
    r = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={'client_id': 'cdse-public', 'grant_type': 'password',
              'username': user, 'password': password},
        timeout=30
    )
    r.raise_for_status()
    return r.json()['access_token']

def download_file(href, token, output_path, chunk_size=1024*1024):
    headers = {'Authorization': f'Bearer {token}'}
    with requests.get(href, headers=headers, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(output_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, desc=output_path.name, leave=False) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))

def get_product_uuid(item):
    href = item.assets['Product'].href
    m = re.search(r'Products\(([^)]+)\)', href)
    return m.group(1) if m else None

def s3_to_odata(s3_url, uuid):
    path = s3_url.replace('s3://eodata/', '')
    parts = path.split('/')
    safe_idx = next(i for i, p in enumerate(parts) if p.endswith('.SAFE'))
    safe_name = parts[safe_idx]
    rest = parts[safe_idx + 1:]
    nodes = '/'.join(f'Nodes({p})' for p in rest)
    return (f'https://download.dataspace.copernicus.eu'
            f'/odata/v1/Products({uuid})/Nodes({safe_name})/{nodes}/$value')

BAND_ASSETS = {
    'B02': 'B02_10m', 'B03': 'B03_10m', 'B04': 'B04_10m', 'B08': 'B08_10m',
    'B8A': 'B8A_20m', 'B11': 'B11_20m', 'B12': 'B12_20m', 'SCL': 'SCL_20m',
}

token = get_cdse_token(CDSE_USER, CDSE_PASSWORD)
token_time = time.time()
skipped, downloaded = [], []

for item in selected:
    if time.time() - token_time > 540:
        token = get_cdse_token(CDSE_USER, CDSE_PASSWORD)
        token_time = time.time()
        print('Token refreshed.')

    uuid = get_product_uuid(item)
    date_str = item.datetime.strftime('%Y%m%d')
    tile_dir = RAW_DIR / item.id / date_str
    tile_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n{item.id[:55]} — {date_str}')

    for band, asset_key in BAND_ASSETS.items():
        if asset_key not in item.assets:
            skipped.append((item.id, band)); continue
        output_path = tile_dir / f'{band}.jp2'
        if output_path.exists():
            print(f'  SKIP  {band}'); continue
        odata_url = s3_to_odata(item.assets[asset_key].href, uuid)
        try:
            download_file(odata_url, token, output_path)
            size_mb = output_path.stat().st_size / 1e6
            print(f'  OK    {band} ({size_mb:.1f} MB)')
            downloaded.append(output_path)
        except Exception as e:
            print(f'  ERROR {band}: {e}')
            if output_path.exists(): output_path.unlink()
            skipped.append((item.id, band))

print(f'\nDownloaded: {len(downloaded)} | Skipped: {len(skipped)}')

In [ ]:
# ── Procesamiento de escenas (mismo pipeline que 04_preprocess.ipynb) ──────────
CRS_UTM = CRS.from_epsg(32720)   # UTM Zone 20S para Córdoba (vs 21S para Corrientes)
BANDS_SPEC = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
SCL_CLEAR  = {4, 5, 6}

def reproject_band(src_path, dst_crs, resolution=10):
    with rasterio.open(src_path) as src:
        crs_src = src.crs if src.crs else CRS.from_epsg(32720)
        transform, width, height = rasterio.warp.calculate_default_transform(
            crs_src, dst_crs, src.width, src.height, *src.bounds, resolution=resolution)
        data = np.zeros((1, height, width), dtype=src.dtypes[0])
        rasterio.warp.reproject(
            source=rasterio.band(src, 1), destination=data[0],
            src_transform=src.transform, src_crs=crs_src,
            dst_transform=transform, dst_crs=dst_crs,
            resampling=rasterio.warp.Resampling.bilinear)
    return data[0], transform

def compute_index(a, b):
    a, b = a.astype(np.float32), b.astype(np.float32)
    denom = a + b
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(denom != 0, (a - b) / denom, 0.0).astype(np.float32)

# Discover downloaded scenes
scenes = []
for item_dir in sorted(RAW_DIR.iterdir()):
    for date_dir in sorted(item_dir.iterdir()):
        if list(date_dir.glob('*.jp2')):
            scenes.append({'item_id': item_dir.name, 'date': date_dir.name, 'dir': date_dir})
            print(f'  {date_dir.name}  {item_dir.name[-25:]}')
print(f'\n{len(scenes)} scenes found')

In [ ]:
# ── Procesar todas las escenas → GeoTIFF ──────────────────────────────────────
processed_paths = []

for scene in scenes:
    scene_dir = scene['dir']
    date      = scene['date']
    item_id   = scene['item_id']
    out_path  = PROC_DIR / f"{item_id[-15:]}_{date}.tif"
    processed_paths.append((scene, out_path))

    if out_path.exists():
        print(f'  SKIP {out_path.name}')
        continue

    print(f'  Processing {date} — {item_id[-25:]}')

    scl_data, scl_transform = reproject_band(scene_dir / 'SCL.jp2', CRS_UTM, 10)
    clear_mask = np.isin(scl_data, list(SCL_CLEAR))
    shape = scl_data.shape

    band_data = {}
    for band in BANDS_SPEC:
        data, _ = reproject_band(scene_dir / f'{band}.jp2', CRS_UTM, 10)
        if data.shape != shape:
            resized = np.zeros(shape, dtype=data.dtype)
            rasterio.warp.reproject(data, resized,
                src_transform=_, src_crs=CRS_UTM,
                dst_transform=scl_transform, dst_crs=CRS_UTM,
                resampling=rasterio.warp.Resampling.bilinear)
            data = resized
        band_data[band] = data

    B08  = band_data['B08'].astype(np.float32)
    NDVI = compute_index(B08, band_data['B04'])
    NBR  = compute_index(B08, band_data['B12'])
    NDWI = compute_index(band_data['B03'], B08)

    stack = np.stack([
        band_data['B02'], band_data['B03'], band_data['B04'],
        band_data['B08'], band_data['B8A'], band_data['B11'], band_data['B12'],
        (NDVI * 10000).astype(np.int16),
        (NBR  * 10000).astype(np.int16),
        (NDWI * 10000).astype(np.int16),
        clear_mask.astype(np.uint8),
    ], axis=0)

    profile = {'driver': 'GTiff', 'dtype': 'int16', 'compress': 'lzw',
               'crs': CRS_UTM, 'transform': scl_transform,
               'width': shape[1], 'height': shape[0],
               'count': 11, 'nodata': -9999}
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(stack)
        dst.update_tags(band_names='B02,B03,B04,B08,B8A,B11,B12,NDVI,NBR,NDWI,MASK',
                        date=date, item_id=item_id, window=scene.get('window_label', ''))

    pct_clear = clear_mask.mean() * 100
    size_mb = out_path.stat().st_size / 1e6
    print(f'  OK   {out_path.name}  ({size_mb:.0f} MB, {pct_clear:.1f}% clear)')

print(f'\n{len(processed_paths)} scenes processed')

In [ ]:
# ── Agrupar escenas procesadas por tile ───────────────────────────────────────
by_tile = defaultdict(lambda: {'pre': None, 'post': None})
for scene, proc_path in processed_paths:
    if not proc_path.exists():
        continue
    # Determinar pre/post desde la fecha de adquisición (no del tag GeoTIFF)
    date = scene['date']  # ej: '20200921' = pre, '20201203' = post
    window_label = 'pre_fire' if date[:6] in ('202008', '202009') else 'post_fire'
    m = re.search(r'_T(\w{5})_', scene['item_id'])
    tile = m.group(1) if m else 'UNKNOWN'
    if 'pre' in window_label:
        by_tile[tile]['pre'] = proc_path
    else:
        by_tile[tile]['post'] = proc_path

print('Tile pairs:')
for tile, paths in by_tile.items():
    pre_ok  = '✓' if paths['pre']  else '✗'
    post_ok = '✓' if paths['post'] else '✗'
    print(f'  {tile}: pre={pre_ok}  post={post_ok}')

In [ ]:
# ── Generar dNBR a nivel de GeoTIFF y guardar como raster ─────────────────────
NBR_BAND  = 9    # 1-indexed en rasterio
MASK_BAND = 11   # 1-indexed en rasterio
DNBR_THRESHOLD = 0.10

dnbr_rasters = {}  # tile -> path del raster dNBR

for tile, paths in by_tile.items():
    if not (paths['pre'] and paths['post']):
        print(f'  {tile}: incomplete pair — skipping')
        continue

    pre_path  = paths['pre']
    post_path = paths['post']
    out_dnbr  = PROC_DIR / f'dnbr_{tile}.tif'
    out_mask  = PROC_DIR / f'burnscar_{tile}.tif'
    dnbr_rasters[tile] = out_mask

    if out_mask.exists():
        print(f'  {tile}: already exists — SKIP')
        continue

    print(f'  {tile}: computing dNBR...')
    with rasterio.open(pre_path)  as src_pre, \
         rasterio.open(post_path) as src_post:

        # Si los tamaños difieren (borde de escena), usar el mínimo
        h = min(src_pre.height,  src_post.height)
        w = min(src_pre.width,   src_post.width)
        win = rasterio.windows.Window(0, 0, w, h)

        nbr_pre  = src_pre.read(NBR_BAND, window=win).astype(np.float32) / 10000.0
        nbr_post = src_post.read(NBR_BAND, window=win).astype(np.float32) / 10000.0
        valid_pre  = src_pre.read(MASK_BAND, window=win) == 1
        valid_post = src_post.read(MASK_BAND, window=win) == 1
        valid = valid_pre & valid_post

        dnbr = np.where(valid, nbr_pre - nbr_post, np.nan)
        burn = (dnbr > DNBR_THRESHOLD).astype(np.uint8)
        burn[~valid] = 255   # nodata

        pct_burned = (burn[valid] == 1).mean() * 100
        print(f'    → {pct_burned:.1f}% burned (dNBR>{DNBR_THRESHOLD})')

        profile = src_pre.meta.copy()
        profile.update(count=1, dtype='uint8', nodata=255,
                       compress='lzw', width=w, height=h,
                       transform=rasterio.windows.transform(win, src_pre.transform))
        with rasterio.open(out_mask, 'w', **profile) as dst:
            dst.write(burn[np.newaxis])
            dst.update_tags(dnbr_threshold=str(DNBR_THRESHOLD),
                            pre_scene=str(pre_path.name),
                            post_scene=str(post_path.name))

print('\ndNBR rasters complete.')

In [ ]:
# ── Extraer patches 256×256 ───────────────────────────────────────────────────
PATCH_SIZE    = 256
STRIDE        = 256
MAX_CLOUD_FRAC = 0.20

IMG_OUT  = PATCH_DIR / 'images'
MASK_OUT = PATCH_DIR / 'masks_dnbr'

total_kept = 0

for tile, burn_path in dnbr_rasters.items():
    post_path = by_tile[tile]['post']
    if not (burn_path.exists() and post_path.exists()):
        continue

    print(f'\nExtracting patches — tile {tile}')
    scene_kept = 0
    prefix = post_path.stem   # usa el stem de la escena post-fuego como nombre

    with rasterio.open(post_path) as src_img, rasterio.open(burn_path) as src_mask:
        H = min(src_img.height, src_mask.height)
        W = min(src_img.width,  src_mask.width)

        for row in range(0, H - PATCH_SIZE + 1, STRIDE):
            for col in range(0, W - PATCH_SIZE + 1, STRIDE):
                window = rasterio.windows.Window(col, row, PATCH_SIZE, PATCH_SIZE)
                img_patch  = src_img.read(window=window)    # (11, 256, 256)
                mask_patch = src_mask.read(1, window=window)  # (256, 256)

                # Banda MASK (índice 10, 0-based) del GeoTIFF = banda 11 en rasterio
                valid = img_patch[10]  # 0-based en el array numpy
                cloud_frac = 1 - valid.mean()
                if cloud_frac > MAX_CLOUD_FRAC:
                    continue

                patch_id = f'{prefix}_r{row:05d}_c{col:05d}'
                np.save(IMG_OUT  / f'{patch_id}.npy', img_patch.astype(np.int16))

                # Convertir nodata (255) a 0 en la máscara
                mask_clean = np.where(mask_patch == 255, 0, mask_patch).astype(np.uint8)
                np.save(MASK_OUT / f'{patch_id}.npy', mask_clean)

                scene_kept += 1

    total_kept += scene_kept
    print(f'  → {scene_kept} patches saved')

img_files  = list(IMG_OUT.glob('*.npy'))
mask_files = list(MASK_OUT.glob('*.npy'))
positive   = sum(1 for f in mask_files if np.load(f, mmap_mode='r').max() > 0)

print(f'\n{"─"*50}')
print(f'Total Córdoba patches: {len(img_files)}')
print(f'Patches with scar    : {positive}/{len(img_files)} ({positive/len(img_files)*100:.1f}% si hay datos)')

In [ ]:
# ── Visualización rápida ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

mask_files = sorted(MASK_OUT.glob('*.npy'))
positive_masks = [f for f in mask_files if np.load(f, mmap_mode='r').max() > 0]

if len(positive_masks) >= 3:
    sample = positive_masks[:3]
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))

    for row, mf in enumerate(sample):
        img  = np.load(IMG_OUT / mf.name)
        mask = np.load(mf)
        nbr  = img[8].astype(np.float32) / 10000.0

        def norm(x):
            v = x[x>0]
            if not len(v): return np.zeros_like(x)
            lo, hi = np.percentile(v, [2,98])
            return np.clip((x-lo)/(hi-lo+1e-6), 0, 1)

        rgb = np.dstack([norm(img[2].astype(np.float32)),
                         norm(img[1].astype(np.float32)),
                         norm(img[0].astype(np.float32))])

        axes[row,0].imshow(rgb)
        axes[row,0].set_title('RGB (B04,B03,B02)', fontsize=9)
        axes[row,0].axis('off')

        axes[row,1].imshow(nbr, cmap='RdYlGn', vmin=-0.5, vmax=0.8)
        axes[row,1].set_title('NBR post-fire', fontsize=9)
        axes[row,1].axis('off')

        axes[row,2].imshow(mask, cmap='Reds', vmin=0, vmax=1)
        burn_pct = mask.mean()*100
        axes[row,2].set_title(f'dNBR burn scar ({burn_pct:.1f}% burned)', fontsize=9)
        axes[row,2].axis('off')

    plt.suptitle('Córdoba Oct 2020 — Test Set', fontsize=12)
    plt.tight_layout()
    out = CORDOBA_DIR / 'cordoba_preview.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print(f'Solo {len(positive_masks)} patches positivos — ajustar BBOX o fechas')

---
**Phase 5 complete → copy `data/cordoba/` to Google Drive, then run `09_evaluate_cordoba.ipynb` in Colab**

```powershell
copy-item -Recurse .\data\cordoba `"G:\Mon Drive\GeoAI\wildfire-spread\data\cordoba`"
```
